[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/01_Tokenizer.ipynb)

# Module 1: Tokenization — The Foundation of LLM Development

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

# Check GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

# Clone repository
if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

# Install dependencies
!pip install -q transformers==4.48.0 datasets==3.6.0 jinja2==3.1.2 tokenizers

print("✅ Setup complete!")

## 📚 Course Introduction

Welcome to **Module 1** of the MiniMind LLM Development Course! In this notebook, we
explore **tokenization** — the process of converting raw text into the numerical
sequences that language models actually understand.

### Learning Objectives

By the end of this module you will be able to:

1. Explain **why** LLMs need a tokenizer and what role it plays in the pipeline.
2. Load and use the **MiniMind tokenizer** (vocab size = 6,400, BPE + ByteLevel).
3. Understand **Byte-Pair Encoding (BPE)** and how it builds a vocabulary.
4. Work with **special tokens** and **chat templates** used in instruction-tuned models.
5. **Train a small tokenizer from scratch** using the HuggingFace `tokenizers` library.

> **Prerequisites:** Basic Python. No ML experience required — we explain everything
> from the ground up.

## 🔤 What is Tokenization?

Language models do not read raw text the way humans do. Before any text can be fed
into a neural network it must be converted to **numbers**. That conversion pipeline
looks like this:

```
Raw text  →  Tokenizer  →  Token IDs  →  Embeddings  →  Model
```

A **tokenizer** is the component that splits text into smaller units called
**tokens** and maps each token to an integer ID. Different strategies exist:

| Strategy | Example for "unhappiness" | Notes |
|---|---|---|
| **Character-level** | `u, n, h, a, p, p, i, n, e, s, s` | Tiny vocab, very long sequences |
| **Word-level** | `unhappiness` | Huge vocab, cannot handle unknown words |
| **Sub-word (BPE)** | `un, happiness` | Best trade-off — small vocab, short sequences |

MiniMind uses **sub-word tokenization** via the **Byte-Pair Encoding (BPE)** algorithm
with a **ByteLevel** pre-tokenizer, giving it a compact vocabulary of just **6,400**
tokens while still handling Chinese, English, code, and special markup.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('./model')

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Total tokens (with special): {len(tokenizer)}")
print(f"Special tokens: {tokenizer.all_special_tokens}")

# Test encoding
text = "Hello, I am MiniMind! 你好，我是MiniMind！"
tokens = tokenizer.encode(text)
print(f"\nText: {text}")
print(f"Token IDs: {tokens}")
print(f"Decoded tokens: {[tokenizer.decode([t]) for t in tokens]}")
print(f"Number of tokens: {len(tokens)}")

## 🧩 Understanding Byte-Pair Encoding (BPE)

BPE is the algorithm behind most modern tokenizers (GPT, LLaMA, MiniMind, …).
Here is how it works, step by step:

1. **Start with bytes.** Every character is first broken down into its UTF-8 byte
   representation (256 possible values). This guarantees *any* text can be encoded.
2. **Count pairs.** Scan the corpus and count how often each *pair* of adjacent
   tokens appears.
3. **Merge the most frequent pair.** Create a new token for that pair and replace
   all occurrences in the corpus.
4. **Repeat** until the desired vocabulary size is reached.

### Concrete Example

```
Corpus:   l o w _ (×5)   l o w e r _ (×2)   n e w e s t _ (×6)

Iteration 1: most frequent pair = (e, s) → merge into "es"
Iteration 2: most frequent pair = (es, t) → merge into "est"
Iteration 3: most frequent pair = (l, o) → merge into "lo"
…and so on until vocab_size is reached.
```

The **ByteLevel** variant used by MiniMind maps each byte to a visible Unicode
character before merging, so the model never encounters raw bytes and every token
is a printable string.

In [ ]:
def visualize_tokens(text, tokenizer):
    """Display tokenization with color-coded token boundaries."""
    ids = tokenizer.encode(text)
    decoded = [tokenizer.decode([t]) for t in ids]
    compression = len(text) / len(ids)

    print(f"Text ({len(text)} chars): {text}")
    print(f"Tokens ({len(ids)}):      {" | ".join(decoded)}")
    print(f"Token IDs:          {ids}")
    print(f"Compression ratio:  {compression:.2f} chars/token")
    print()

# ── English ──
visualize_tokens("Large language models are fascinating!", tokenizer)

# ── Chinese ──
visualize_tokens("大型语言模型非常有趣！", tokenizer)

# ── Code ──
visualize_tokens("def hello(): return 42", tokenizer)

# ── Mixed Chinese/English ──
visualize_tokens("Python 是一种高级编程语言", tokenizer)

# ── Compression ratio comparison ──
print("=" * 60)
print("Compression Ratio Comparison")
print("=" * 60)
test_texts = [
    ("English",  "Artificial intelligence is transforming how we live and work."),
    ("Chinese",  "人工智能正在改变我们的生活和工作方式。"),
    ("Code",     "for i in range(10): print(f\'value={i}\')"),
    ("Mixed",    "使用 transformer 架构训练 LLM 大模型"),
]
for label, t in test_texts:
    ids = tokenizer.encode(t)
    ratio = len(t) / len(ids)
    print(f"  {label:8s} | Chars: {len(t):3d} | Tokens: {len(ids):3d} | Ratio: {ratio:.2f}")

## 🏷️ Special Tokens and Chat Templates

In addition to the regular vocabulary learned by BPE, MiniMind defines **36 special
tokens** that carry structural meaning:

| Category | Tokens | Purpose |
|---|---|---|
| **Core** | `<|im_start|>`, `<|im_end|>`, `<|endoftext|>` | Mark message boundaries |
| **Reasoning** | `<think>`, `</think>` | Wrap chain-of-thought reasoning |
| **Tool use** | `<tool_call>`, `</tool_call>`, `<tool_response>`, `</tool_response>` | Function calling |
| **Vision / Audio** | `<|vision_start|>`, `<|audio_pad|>`, … | Multimodal placeholders |
| **Buffer** | `<|buffer1|>` – `<|buffer9|>` | Reserved for future use |

These tokens are **never split** by the tokenizer — they are always treated as
single, atomic units.

### Chat Templates

Modern instruction-tuned models expect conversations formatted in a specific way.
MiniMind uses the **ChatML** style:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
<think>
…reasoning…
</think>

The answer is 4.<|im_end|>
```

The tokenizer stores this format as a **Jinja2 chat template** so you can apply it
automatically — let's see how.

In [ ]:
# Show how chat templates work
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
]

formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("Formatted chat template:")
print(formatted)
print("\nToken IDs:")
print(tokenizer.encode(formatted))

In [ ]:
# Multi-turn conversation with assistant response
multi_turn = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
]

formatted_multi = tokenizer.apply_chat_template(
    multi_turn, tokenize=False, add_generation_prompt=True
)
print("Multi-turn formatted output:")
print(formatted_multi)

# Verify round-trip consistency
ids = tokenizer.encode(formatted_multi)
decoded = tokenizer.decode(ids, skip_special_tokens=False)
print(f"\nRound-trip consistent: {decoded == formatted_multi}")
print(f"Encoded length: {len(ids)} tokens")

## 🔨 Training a Tokenizer from Scratch

The MiniMind repo ships with a pre-trained tokenizer, but understanding *how* it was
built is essential. The original training script (`trainer/train_tokenizer.py`) does
the following:

1. **Loads training data** — the first 10,000 lines of `dataset/sft_t2t_mini.jsonl`
   (a bilingual Chinese/English conversation dataset).
2. **Initializes a BPE tokenizer** with a `ByteLevel` pre-tokenizer.
3. **Defines special tokens** — 36 tokens covering chat markup, tool calling,
   reasoning, multimodal placeholders, and buffer slots.
4. **Trains** the tokenizer to a vocabulary of **6,400** tokens.
5. **Saves** `tokenizer.json` and `tokenizer_config.json`.

Below we reproduce a **simplified** version that trains a tiny BPE tokenizer from a
handful of sample sentences so you can see the full lifecycle without needing the
full dataset.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# Create a small training corpus
sample_texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Large language models use transformers architecture.",
    "Tokenization converts text into numerical representations.",
    "Byte-Pair Encoding is the most popular sub-word algorithm.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning models require large amounts of training data.",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Transfer learning lets models trained on one task adapt to another.",
    "The transformer architecture revolutionized natural language processing.",
    "Pre-training on large corpora gives language models general knowledge.",
    "人工智能是计算机科学的一个重要分支。",
    "大型语言模型使用了注意力机制。",
    "自然语言处理让计算机能够理解人类语言。",
    "深度学习需要大量的训练数据。",
    "分词是将文本转换为数字表示的过程。",
]

# Save to file for the trainer
corpus_path = "sample_corpus.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for text in sample_texts:
        f.write(text + "\n")

# Build a small BPE tokenizer
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()

trainer_obj = trainers.BpeTrainer(
    vocab_size=500,
    special_tokens=["<s>", "</s>", "<pad>", "<unk>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)

tok.train([corpus_path], trainer=trainer_obj)

# Test the freshly trained tokenizer
test_text = "Machine learning models are powerful"
encoded = tok.encode(test_text)
print(f"Vocab size: {tok.get_vocab_size()}")
print(f"\nText: {test_text}")
print(f"Tokens: {encoded.tokens}")
print(f"IDs:    {encoded.ids}")
print(f"Decoded: {tok.decode(encoded.ids)}")

# Clean up
import os
os.remove(corpus_path)

In [ ]:
# Compare our toy tokenizer with MiniMind's tokenizer
comparison_text = "Language models learn from data"

mini_ids = tokenizer.encode(comparison_text)
toy_enc  = tok.encode(comparison_text)

print(f"Text: {comparison_text}\n")
print(f"MiniMind tokenizer (vocab={tokenizer.vocab_size}):")
print(f"  Tokens: {[tokenizer.decode([t]) for t in mini_ids]}")
print(f"  Count:  {len(mini_ids)}")
print(f"\nToy tokenizer (vocab={tok.get_vocab_size()}):")
print(f"  Tokens: {toy_enc.tokens}")
print(f"  Count:  {len(toy_enc.ids)}")
print(f"\nA larger vocabulary generally means better compression (fewer tokens).")

## ✏️ Exercises

Try these to deepen your understanding:

1. **Vocabulary exploration** — Pick 10 random IDs between 36 and 6,399 and decode
   them. What kinds of tokens do you find? Are they whole words, sub-words, or
   individual characters?

2. **Language comparison** — Tokenize the same sentence in English and Chinese. Which
   language produces fewer tokens? Why?

3. **Vocab size experiment** — Re-run the toy tokenizer training with `vocab_size` set
   to 200, 500, and 1,000. How does vocab size affect the number of tokens produced
   for the same input?

4. **Special tokens** — Add a new special token `<my_token>` to the MiniMind
   tokenizer with `tokenizer.add_special_tokens({"additional_special_tokens":
   ["<my_token>"]})`. Verify that it gets its own token ID and is never split.

5. **Chat template** — Create a multi-turn conversation with tool use and apply the
   chat template. Examine how `<tool_call>` and `<tool_response>` tags appear in the
   formatted output.

In [ ]:
# Exercise 1: Vocabulary exploration
import random
random.seed(42)
sample_ids = random.sample(range(36, tokenizer.vocab_size), 10)
print("Random vocabulary samples:")
for tid in sample_ids:
    token_str = tokenizer.decode([tid])
    print(f"  ID {tid:5d} → {repr(token_str)}")

# Try the other exercises below!


## 📝 Summary

In this module we covered:

- **Why tokenization matters** — LLMs operate on numbers, not raw text.
- **Byte-Pair Encoding** — The algorithm that builds a sub-word vocabulary by
  iteratively merging the most frequent pairs.
- **The MiniMind tokenizer** — 6,400 tokens, BPE + ByteLevel, 36 special tokens
  covering chat, reasoning, tool use, and multimodal features.
- **Chat templates** — How conversations are formatted with `<|im_start|>` /
  `<|im_end|>` markers and `<think>` tags for chain-of-thought.
- **Training a tokenizer from scratch** — Using the HuggingFace `tokenizers`
  library to build your own BPE tokenizer.

### 🔜 Next Steps

In **Module 2** we will dive into the **model architecture** — how MiniMind
turns token embeddings into predictions using attention, feed-forward layers,
and the full transformer stack.

---
*MiniMind LLM Development Course — Module 1: Tokenization*